# Lib


In [4]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Data


In [5]:
df = pd.read_csv("../../data/processed/processed.csv")

In [6]:
df = df.drop(columns=["name"], errors="ignore")

In [7]:
df.head()

,source,target,ml_target
0,0,23977,0
1,1,34526,0
2,1,2370,0
3,1,14683,0
4,1,29982,0


# Graph


In [8]:
G = nx.from_pandas_edgelist(df, source="source", target="target", create_using=nx.Graph())

In [9]:
nx.describe(G)

Number of nodes                : 37700
Number of edges                : 289003
Directed                       : False
Multigraph                     : False
Tree                           : False
Bipartite                      : False
Average degree (min, max)      : 15.33 (1, 9458)
Number of connected components : 1


In [10]:
# Convert edges to sorted tuples
all_edges = [tuple(sorted((u, v))) for u, v in G.edges()]

# Remove duplicates while preserving order
all_edges = list(dict.fromkeys(all_edges))

rng = np.random.default_rng(42)

print(f"Total unique edges: {len(all_edges):,}")

Total unique edges: 289,003


# Sampling


## Positive Edge Sampling

The true edges of the graph are split into two subsets:
- Train Positive Edges
- Test Positive Edges

Note:
- Avoid assigning `bridge edges` to the test set.
- A **bridge edge** is an edge whose removal increases the number of connected components in the graph.

In [11]:
def sample_positive_edges(
    edges: list[tuple[int, int]],
    test_ratio: float,
    graph: nx.Graph,
    random_state: int,
) -> tuple[
    list[tuple[int, int]],
    list[tuple[int, int]],
    set[tuple[int, int]],
]:

    rng_local = np.random.default_rng(random_state)

    n_test = int(len(edges) * test_ratio)

    # Find bridge edges
    bridge_edges = {tuple(sorted(edge)) for edge in nx.bridges(graph)}

    # Prefer non-bridge edges for test
    non_bridge_edges = [edge for edge in edges if edge not in bridge_edges]

    if len(non_bridge_edges) >= n_test:

        test_indices = rng_local.choice(
            len(non_bridge_edges),
            size=n_test,
            replace=False,
        )

        test_pos = [non_bridge_edges[index] for index in test_indices]

    else:
        test_pos = list(non_bridge_edges)

        remaining = n_test - len(test_pos)

        bridge_pool = [edge for edge in edges if edge in bridge_edges]

        if remaining > len(bridge_pool):
            raise ValueError(
                "Not enough positive edges to create the requested test split."
            )

        bridge_indices = rng_local.choice(
            len(bridge_pool),
            size=remaining,
            replace=False,
        )

        test_pos.extend([bridge_pool[index] for index in bridge_indices])

    test_pos = [tuple(edge) for edge in test_pos]

    test_set = set(test_pos)

    train_pos = [edge for edge in edges if edge not in test_set]

    return train_pos, test_pos, bridge_edges

## Negative Sampling

- A **negative** is a node pair $(A, B)$ that does **not** have a direct connection.
- Negatives are sampled uniformly from non-edge pairs using rejection sampling.

In [12]:
def sample_negative_pairs(
    graph: nx.Graph,
    n_samples: int,
    forbidden_pairs: set[tuple[int, int]],
    random_state: int,
) -> list[tuple[int, int]]:

    rng_local = np.random.default_rng(random_state)

    nodes = list(graph.nodes())
    if len(nodes) < 2:
        raise ValueError("Graph must contain at least two nodes.")

    sampled_pairs = []
    sampled_set = set()
    max_attempts = max(n_samples * 20, 1000)
    attempts = 0

    while len(sampled_pairs) < n_samples and attempts < max_attempts:
        left_index, right_index = rng_local.choice(len(nodes), size=2, replace=False)
        u = nodes[left_index]
        v = nodes[right_index]
        pair = tuple(sorted((u, v)))

        if pair in forbidden_pairs or pair in sampled_set or graph.has_edge(u, v):
            attempts += 1
            continue

        sampled_set.add(pair)
        sampled_pairs.append(pair)
        attempts += 1

    if len(sampled_pairs) < n_samples:
        raise ValueError("Not enough negative pairs available for the requested sample size.")

    return sampled_pairs

# Splitting


## Train/test samples


In [13]:
train_pos, test_pos, bridge_edges = sample_positive_edges(
    edges=all_edges,
    test_ratio=0.2,
    graph=G,
    random_state=42,
)

print(f"Train positive edges: {len(train_pos):,}")
print(f"Test positive edges: {len(test_pos):,}")

Train positive edges: 231,203
Test positive edges: 57,800


## Negative samples


In [14]:
positive_forbidden = set(all_edges)

train_neg = sample_negative_pairs(
    graph=G,
    n_samples=len(train_pos),
    forbidden_pairs=positive_forbidden,
    random_state=42,
)

negative_forbidden = positive_forbidden.union(train_neg)

test_neg = sample_negative_pairs(
    graph=G,
    n_samples=len(test_pos),
    forbidden_pairs=negative_forbidden,
    random_state=42,
)

train_neg_set = set(train_neg)
test_neg_set = set(test_neg)

assert train_neg_set.isdisjoint(positive_forbidden)
assert test_neg_set.isdisjoint(positive_forbidden)
assert train_neg_set.isdisjoint(test_neg_set)

In [15]:
print(f"Train negative edges: {len(train_neg):,}")
print(f"Test negative edges: {len(test_neg):,}")

Train negative edges: 231,203
Test negative edges: 57,800


# Link Prediction Dataset

Dataset gồm:

- source
- target
- label
  - 1 = positive edge
  - 0 = negative edge
- split
  - train/test
- type
  - positive/negative


In [16]:
train_df = pd.DataFrame(
    {
        "source": [u for u, _ in train_pos + train_neg],
        "target": [v for _, v in train_pos + train_neg],
        "label": [1] * len(train_pos) + [0] * len(train_neg),
        "split": ["train"] * (len(train_pos) + len(train_neg)),
        "type": (["positive"] * len(train_pos) + ["negative"] * len(train_neg)),
    }
)

test_df = pd.DataFrame(
    {
        "source": [u for u, _ in test_pos + test_neg],
        "target": [v for _, v in test_pos + test_neg],
        "label": [1] * len(test_pos) + [0] * len(test_neg),
        "split": ["test"] * (len(test_pos) + len(test_neg)),
        "type": (["positive"] * len(test_pos) + ["negative"] * len(test_neg)),
    }
)

In [17]:
df_indexed = df.set_index(["source", "target"])

train_df = train_df.join(df_indexed, on=["source", "target"], how="left")
test_df = test_df.join(df_indexed, on=["source", "target"], how="left")

# Distribution

In [18]:
label_distribution_train = (train_df.groupby(["split", "label"]).size().reset_index(name="count"))
label_distribution_train["percentage"] = (label_distribution_train["count"] / label_distribution_train.groupby("split")["count"].transform("sum") * 100).round(2)
label_distribution_train

,split,label,count,percentage
0,train,0,231203,50.0
1,train,1,231203,50.0


In [19]:
label_distribution_train = (test_df.groupby(["split", "label"]).size().reset_index(name="count"))
label_distribution_train["percentage"] = (label_distribution_train["count"] / label_distribution_train.groupby("split")["count"].transform("sum") * 100).round(2)
display(label_distribution_train)

,split,label,count,percentage
0,test,0,57800,50.0
1,test,1,57800,50.0


In [20]:
print(f"Total edges in graph: {len(all_edges):,}")
print(f"Bridge edges protected in train: {len(bridge_edges):,}")

print(f"Train positive: {len(train_pos):,}")
print(f"Train negative: {len(train_neg):,}")

print(f"Test positive: {len(test_pos):,}")
print(f"Test negative: {len(test_neg):,}")

Total edges in graph: 289,003
Bridge edges protected in train: 5,241
Train positive: 231,203
Train negative: 231,203
Test positive: 57,800
Test negative: 57,800


# Export


In [21]:
train_df = train_df.drop(columns=["split", "type"], errors="ignore")
test_df = test_df.drop(columns=["split", "type"], errors="ignore")

In [22]:
train_df.head(), test_df.head()

(   source  target  label  ml_target
 0       0   23977      1        0.0
 1      69   23977      1        0.0
 2    1966   23977      1        0.0
 3    4422   23977      1        1.0
 4    5631   23977      1        0.0,
    source  target  label  ml_target
 0   10402   24705      1        0.0
 1    1906   35876      1        0.0
 2    2431    4540      1        NaN
 3    2834   25177      1        1.0
 4   27521   29982      1        0.0)

In [23]:
train_df.to_csv(
    "../../data/completed/train.csv",
    index=False,
)
test_df.to_csv(
    "../../data/completed/test.csv",
    index=False,
)